<a href="https://colab.research.google.com/github/darrickpang/Email/blob/master/NMT_project_Darrick_Pang_original.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datasets import load_dataset
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
import time
import pandas as pd
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [ ]:
results = []
epochs = 20
batch_size = 64
training_samples = 1000000
model = "Transformer"
range_bleu = 200

In [ ]:
pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00


In [ ]:
pip install sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 5.3 MB/s eta 0:00:00


In [ ]:
from evaluate import load
bleu = load("sacrebleu")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
dataset = load_dataset("wmt16", "de-en")

train_data = dataset["train"].select(range(training_samples))
test_data = dataset["test"]
val_data = dataset["validation"]

README.md: 0.00B [00:00, ?B/s]

de-en/train-00000-of-00003.parquet:   0%|          | 0.00/282M [00:00<?, ?B/s]

de-en/train-00001-of-00003.parquet:   0%|          | 0.00/267M [00:00<?, ?B/s]

de-en/train-00002-of-00003.parquet:   0%|          | 0.00/277M [00:00<?, ?B/s]

de-en/validation-00000-of-00001.parquet:   0%|          | 0.00/343k [00:00<?, ?B/s]

de-en/test-00000-of-00001.parquet:   0%|          | 0.00/475k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4548885 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2169 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2999 [00:00<?, ? examples/s]

In [ ]:
print(train_data["translation"])
print(val_data)

Column([{'de': 'Wiederaufnahme der Sitzungsperiode', 'en': 'Resumption of the session'}, {'de': 'Ich erkläre die am Freitag, dem 17. Dezember unterbrochene Sitzungsperiode des Europäischen Parlaments für wiederaufgenommen, wünsche Ihnen nochmals alles Gute zum Jahreswechsel und hoffe, daß Sie schöne Ferien hatten.', 'en': 'I declare resumed the session of the European Parliament adjourned on Friday 17 December 1999, and I would like once again to wish you a happy new year in the hope that you enjoyed a pleasant festive period.'}, {'de': 'Wie Sie feststellen konnten, ist der gefürchtete "Millenium-Bug " nicht eingetreten. Doch sind Bürger einiger unserer Mitgliedstaaten Opfer von schrecklichen Naturkatastrophen geworden.', 'en': "Although, as you will have seen, the dreaded 'millennium bug' failed to materialise, still the people in a number of countries suffered a series of natural disasters that truly were dreadful."}, {'de': 'Im Parlament besteht der Wunsch nach einer Aussprache im V

In [ ]:
source_text = [german["de"] for german in train_data["translation"]]
# print(source_text)

target_text = [english["en"] for english in train_data["translation"]]
# print(target_text)

# Add start and end tokens to target text
target_text_with_tokens = ['<start> ' + text + ' <end>' for text in target_text]


source_tokenizer = Tokenizer(num_words=30000, filters='')
target_tokenizer = Tokenizer(num_words=30000, filters='')

source_tokenizer.fit_on_texts(source_text)
target_tokenizer.fit_on_texts(target_text_with_tokens)

source_sequence = source_tokenizer.texts_to_sequences(source_text)
target_sequence = target_tokenizer.texts_to_sequences(target_text_with_tokens)

max_src_len = 40
max_tgt_len = 40
encoder_input = pad_sequences(source_sequence, maxlen=max_src_len, padding='post')
decoder_input = pad_sequences([s[:-1] for s in target_sequence], maxlen=max_tgt_len, padding='post')
decoder_target = pad_sequences([s[1:] for s in target_sequence], maxlen=max_tgt_len, padding='post')

In [ ]:
source_vocab = 30000
target_vocab = 30000
embedding_dim = 256
latent_dim = 512

# encoder_inputs = layers.Input(shape=(max_src_len,))
# encoder_embeddings = layers.Embedding(source_vocab, embedding_dim)(encoder_inputs)
# encoder_lstm = layers.Bidirectional(layers.LSTM(latent_dim, return_sequences=True, return_state=True))
# encoder_outputs, forward_h, forward_c, backward_h, backward_c = encoder_lstm(encoder_embeddings)

# # Concatenate forward and backward states
# state_h = layers.Concatenate()([forward_h, backward_h])
# state_c = layers.Concatenate()([forward_c, backward_c])


# decoder_inputs = layers.Input(shape=(1,))
# decoder_embeddings = layers.Embedding(target_vocab, embedding_dim)(decoder_inputs)
# decoder_lstm = layers.LSTM(latent_dim * 2, return_sequences=True, return_state=True)
# decoder_outputs, _, _ = decoder_lstm(decoder_embeddings, initial_state=[state_h, state_c])

# attention = layers.Attention()([decoder_outputs, encoder_outputs])
# decoder_concat = layers.Concatenate(axis=-1)([decoder_outputs, attention])

# # Add a Dense layer for outputting probabilities for each word in the target vocabulary
# decoder_dense = layers.Dense(target_vocab, activation='softmax')
# decoder_outputs = decoder_dense(decoder_concat)


# model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
# model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

# ---- Hyperparams (small, fast) ----
d_model = 256          # model width
num_heads = 4
d_ff = 1024            # FFN hidden
num_enc = 3            # encoder layers
num_dec = 3            # decoder layers
dropout_rate = 0.1
v_src = source_vocab   # 30_000 from your code
v_tgt = target_vocab   # 30_000 from your code

# ---- Positional Encoding ----
class PositionalEncoding(layers.Layer):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        import numpy as np
        pe = np.zeros((max_len, d_model), dtype="float32")
        position = np.arange(0, max_len)[:, None]
        div = np.exp(np.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = np.sin(position * div)
        pe[:, 1::2] = np.cos(position * div)
        self.pe = tf.constant(pe[None, ...])   # [1, max_len, d_model]
    def call(self, x):
        return x + self.pe[:, :tf.shape(x)[1], :]

# ---- Masks ----
def padding_mask(x):
    # x: [B, T] int32
    return tf.cast(tf.equal(x, 0), tf.bool)  # True where PAD

def causal_mask(T):
    return tf.linalg.band_part(tf.ones((T, T), dtype=tf.bool), -1, 0)  # lower-triangular True

# ---- Encoder/Decoder Blocks ----
def encoder_block(x, pad_mask):
    # x: [B, T, d_model]
    attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model, dropout=dropout_rate)
    y = attn(query=x, value=x, key=x, attention_mask=~pad_mask[:, None, None, :])  # mask True=keep
    x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(dropout_rate)(y))
    y = layers.Dense(d_ff, activation="relu")(x)
    y = layers.Dense(d_model)(y)
    x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(dropout_rate)(y))
    return x

def decoder_block(x, enc_out, look_mask, enc_pad_mask):
    # 1️⃣ Self-attention
    self_attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model, dropout=dropout_rate)
    y = self_attn(query=x, value=x, key=x, attention_mask=look_mask[:, None, :, :])
    x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(0.1)(y))

    # 2️⃣ Cross-attention (encoder–decoder)
    cross_attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model, dropout=dropout_rate)
    y = cross_attn(query=x, value=enc_out, key=enc_out, attention_mask=~enc_pad_mask[:, None, None, :])
    x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(0.1)(y))

    # 3️⃣ Feed-forward
    y = layers.Dense(d_ff, activation="relu")(x)
    y = layers.Dense(d_model)(y)
    x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(0.1)(y))
    return x


# ---- Inputs (reuse your max_src_len / max_tgt_len) ----
enc_inp = layers.Input(shape=(max_src_len,), name="enc_tokens")
dec_inp = layers.Input(shape=(max_tgt_len,), name="dec_tokens")  # teacher-forced full sequence

# Embeddings (+ tie dims)
enc_emb = layers.Embedding(v_src, d_model, mask_zero=True)(enc_inp)
dec_emb = layers.Embedding(v_tgt, d_model, mask_zero=True)(dec_inp)

# Add positional encodings
enc_x = PositionalEncoding(d_model)(enc_emb)
dec_x = PositionalEncoding(d_model)(dec_emb)

# Masks
enc_pad = layers.Lambda(lambda x: tf.cast(tf.equal(x, 0), tf.bool), name="enc_pad")(enc_inp)

# Decoder look-ahead + padding mask
def make_lookahead_mask(x):
    seq_len = tf.shape(x)[1]
    mask = tf.cast(tf.not_equal(x, 0), tf.bool)
    mask = tf.logical_and(tf.tile(mask[:, None, :], [1, seq_len, 1]),
                          tf.linalg.band_part(tf.ones((seq_len, seq_len), dtype=tf.bool), -1, 0))
    return mask

look = layers.Lambda(make_lookahead_mask, name="lookahead_mask")(dec_inp)

# Encoder stack
for _ in range(num_enc):
    enc_x = encoder_block(enc_x, enc_pad)

# Decoder stack
x = dec_x
for _ in range(num_dec):
    x = decoder_block(x, enc_x, look, enc_pad)

# Output projection
logits = layers.Dense(v_tgt, activation="softmax")(x)

transformer = Model([enc_inp, dec_inp], logits)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:965: UserWarning: Layer 'positional_encoding' (of type PositionalEncoding) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:965: UserWarning: Layer 'positional_encoding_1' (of type PositionalEncoding) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


In [ ]:
# Label smoothing (improves BLEU a bit)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=False,
    # label_smoothing=0.1 # Removed unsupported argument
)

from tensorflow.keras import backend as K

def smoothed_sparse_categorical_crossentropy(y_true, y_pred, label_smoothing=0.1):
    y_true = tf.cast(y_true, tf.int32)
    num_classes = tf.shape(y_pred)[-1]
    y_true_one_hot = tf.one_hot(y_true, depth=num_classes)
    smooth_positives = 1.0 - label_smoothing
    smooth_negatives = label_smoothing / tf.cast(num_classes, tf.float32)
    y_true_smooth = y_true_one_hot * smooth_positives + smooth_negatives
    loss = -tf.reduce_sum(y_true_smooth * tf.math.log(y_pred + 1e-7), axis=-1)
    return tf.reduce_mean(loss)

# Noam-style schedule (Transformer baseline)
class NoamSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, d_model, warmup_steps=4000):
        self.d_model = tf.cast(d_model, tf.float32)
        self.warmup_steps = warmup_steps
    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        return (self.d_model ** -0.5) * tf.minimum(step ** -0.5, step * (self.warmup_steps ** -1.5))

lr = NoamSchedule(d_model, warmup_steps=4000)
opt = tf.keras.optimizers.Adam(learning_rate=lr, beta_1=0.9, beta_2=0.98, epsilon=1e-9)

transformer.compile(optimizer=opt, loss=smoothed_sparse_categorical_crossentropy, metrics=["accuracy"])

In [ ]:
start_time = time.time()

early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1)

transformer.fit(
    [encoder_input, decoder_input],
    decoder_target,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.1,
    callbacks=[early_stop, reduce_lr]
)

end_time = time.time()
elapsed = end_time - start_time
print(f"Training time: {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")

Epoch 1/20
 2046/14063 ━━━━━━━━━━━━━━━━━━━━ 32:16 161ms/step - accuracy: 0.4152 - loss: 6.5646

In [ ]:
# start_time = time.time()

# model.fit([encoder_input, decoder_input], np.expand_dims(decoder_target, -1), batch_size=batch_size, epochs=epochs, validation_split=0.1)

# end_time = time.time()
# elapsed = end_time - start_time
# print(f"Training time: {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")

Epoch 1/10
2813/2813 ━━━━━━━━━━━━━━━━━━━━ 617s 218ms/step - accuracy: 0.5120 - loss: 3.5084 - val_accuracy: 0.6056 - val_loss: 2.5216
Epoch 2/10
2813/2813 ━━━━━━━━━━━━━━━━━━━━ 616s 219ms/step - accuracy: 0.6459 - loss: 2.0829 - val_accuracy: 0.6380 - val_loss: 2.1881
Epoch 3/10
2813/2813 ━━━━━━━━━━━━━━━━━━━━ 614s 218ms/step - accuracy: 0.6976 - loss: 1.5406 - val_accuracy: 0.6419 - val_loss: 2.1619
Epoch 4/10
2813/2813 ━━━━━━━━━━━━━━━━━━━━ 614s 218ms/step - accuracy: 0.7463 - loss: 1.1811 - val_accuracy: 0.6403 - val_loss: 2.2302
Epoch 5/10
2813/2813 ━━━━━━━━━━━━━━━━━━━━ 615s 219ms/step - accuracy: 0.7853 - loss: 0.9433 - val_accuracy: 0.6361 - val_loss: 2.3297
Epoch 6/10
2813/2813 ━━━━━━━━━━━━━━━━━━━━ 614s 218ms/step - accuracy: 0.8179 - loss: 0.7681 - val_accuracy: 0.6325 - val_loss: 2.4356
Epoch 7/10
2813/2813 ━━━━━━━━━━━━━━━━━━━━ 614s 218ms/step - accuracy: 0.8443 - loss: 0.6361 - val_accuracy: 0.6286 - val_loss: 2.5524
Epoch 8/10
2813/2813 ━━━━━━━━━━━━━━━━━━━━ 614s 218ms/step - ac

In [ ]:
# encoder_inference_model = Model(encoder_inputs, [encoder_outputs, state_h, state_c])

# decoder_state_input_h = layers.Input(shape=(latent_dim * 2,)) # Corrected shape
# decoder_state_input_c = layers.Input(shape=(latent_dim * 2,)) # Corrected shape
# encoder_outputs_input = layers.Input(shape=(max_src_len, latent_dim*2)) # Input for encoder outputs, shape should match encoder_outputs


# decoder_output, state_h_inf, state_c_inf = decoder_lstm(
#     decoder_embeddings, initial_state=[decoder_state_input_h, decoder_state_input_c]
# )

# attention_inf = layers.Attention()([decoder_output, encoder_outputs_input])
# decoder_concat_inf = layers.Concatenate(axis=-1)([decoder_output, attention_inf])


# decoder_output_inf = decoder_dense(decoder_concat_inf)

# decoder_model = Model(
#     [decoder_inputs, decoder_state_input_h, decoder_state_input_c, encoder_outputs_input],
#     [decoder_output_inf, state_h_inf, state_c_inf]
# )

In [ ]:
def translate(sentence, beam_width=4, max_len=max_tgt_len, alpha=0.6):
    # ---- Encode the source sentence ----
    src_seq = source_tokenizer.texts_to_sequences([sentence])
    src_seq = pad_sequences(src_seq, maxlen=max_src_len, padding='post')

    start_id = target_tokenizer.word_index['<start>']
    end_id   = target_tokenizer.word_index['<end>']

    # Each beam is (sequence_so_far, cumulative_log_prob)
    beams = [([start_id], 0.0)]

    for _ in range(max_len):
        new_beams = []
        for seq, score in beams:
            # Stop expanding finished hypotheses
            if seq[-1] == end_id:
                new_beams.append((seq, score))
                continue

            dec_seq = pad_sequences([seq], maxlen=max_tgt_len, padding='post')
            preds = transformer.predict([src_seq, dec_seq], verbose=0)
            probs = preds[0, len(seq)-1, :]  # distribution for next token

            # pick top-k candidates
            top_ids = np.argsort(probs)[-beam_width:]
            for t in top_ids:
                new_seq = seq + [int(t)]
                new_score = score + np.log(probs[t] + 1e-9)
                new_beams.append((new_seq, new_score))

        # Keep the best `beam_width` beams
        # ---- Length normalization function ----
        def length_norm(score, length, alpha=alpha):
            return score / ((5 + length) / 6) ** alpha

        # Keep the best normalized beams
        beams = sorted(
            new_beams,
            key=lambda x: length_norm(x[1], len(x[0])),
            reverse=True
        )[:beam_width]

        # Early-stop if all beams ended
        if all(seq[-1] == end_id for seq, _ in beams):
            break

    # Take best-scoring beam
    best_seq = beams[0][0]
    words = [target_tokenizer.index_word.get(i, '') for i in best_seq[1:] if i not in (0, end_id)]
    return ' '.join(words)

In [ ]:
print(translate("Das ist ein Test."))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 196ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
that is a crucial way of doing things.


In [ ]:
# Generate predictions on a small subset of validation data
predictions = []
references = []

for i in range(range_bleu):  # 200 sentences for demo, increase later
    de_sentence = test_data[i]["translation"]["de"]
    en_reference = test_data[i]["translation"]["en"]

    en_predicted = translate(de_sentence, beam_width=6, alpha=0.7)

    predictions.append(en_predicted)
    references.append([en_reference])  # sacreBLEU expects list of list

result = bleu.compute(predictions=predictions, references=references)
print(f"BLEU score: {result['score']:.2f}")

BLEU score: 8.97


In [ ]:
results.append({"Epochs": epochs, "Batch Size": batch_size, "Training Time": elapsed, "BLEU": result['score'], "Training Data": training_samples, "Model": model, "Range BLEU": range_bleu})

In [ ]:
df = pd.DataFrame(results)
df

,Epochs,Batch Size,Training Time,BLEU,Training Data,Model,Range BLEU
0,20,64,6993.629845,8.974931,1000000,Transformer,200


In [ ]:
df.to_excel('NMT_output_Darrick_Pang.xlsx', sheet_name='MyData')

In [ ]:
df = pd.read_excel('NMT_output_Darrick_Pang.xlsx')
df

,Unnamed: 0,Epochs,Batch Size,Training Time,BLEU,Training Data,Model,Range BLEU
0,0,20,64,6993.629845,8.974931,1000000,Transformer,200


In [ ]:
!jupyter nbconvert --clear-output --to notebook --output="NMT_project_Darrick_Pang.ipynb" "NMT_project_Darrick_Pang_original.ipynb"